# Notebook 18: Diagnostic-Selected Object-Masked PFB + SAC

This notebook implements one improved variant derived from notebook 18 Variant 3.

The difference is that object-injection scales are **not manually chosen** as `3,4,5` or `3,6,9`. Instead, the notebook first runs a region-aware diagnostic over 50 CSD100 image pairs:

```text
ObjectInjectionScore(s) = ΔObjectStyle(s) - λΔBackgroundStyle(s) - μΔStyleObjectLeakage(s)
```

Then it selects the top-k candidate scales and uses them for generation on the fixed 20-pair benchmark.

Final improved variant:

```text
Global style:       scales 0,1,2, rank=1, decay
Object style:       diagnostic top-k scales, soft/dilated object mask, foreground rank=2
Background leakage: small background strength 0.15 for smooth style continuity
SAC:                enabled from scale 2 onward
```

The notebook outputs the generated samples, scale trajectory diagnostics, and CLIP style metrics including `S_harmonic`.


In [ ]:
from pathlib import Path
import gc
import importlib.util
import os
import re
import shutil
import subprocess
import sys
import time

# All files are kept under one directory so the notebook can be rerun safely.
ROOT = Path('/content/notebook_18_diagnostic_selected_object_masked_pfb_sac')
PORT_DIR = ROOT / 'gguf_port'
OFFICIAL_DIR = PORT_DIR / 'Infinity'
ASSET_DIR = ROOT / 'assets'
OUTPUT_DIR = ROOT / 'outputs'
for path in (PORT_DIR, ASSET_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPO = 'https://github.com/FoundationVision/Infinity.git'
GGUF_REPO = 'kzopp/Infinity-2B-GGUF_UNOFFICIAL'
MODEL_PN = '0.25M'  # Colab-friendly 512px. Change to '1M' only on a larger GPU runtime.
CFG_SCALE = 1.0
TAU = 0.1
SEED = 2026
T5_DEVICE = 'cuda'  # T4/Colab: keep T5 off host RAM; use 'cpu' only with a high-RAM runtime.
print('ROOT:', ROOT)
print('MODEL_PN:', MODEL_PN, '| CFG:', CFG_SCALE, '| TAU:', TAU, '| SEED:', SEED, '| T5:', T5_DEVICE)


In [ ]:
# Check the runtime before installing anything.
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('VRAM GiB:', round(props.total_memory / 2**30, 2))
else:
    print('WARNING: no GPU detected; inference will be extremely slow.')


In [ ]:
# Install the packages used by the GGUF loader.
# We intentionally do not install torch or flash-attn here: Colab already ships torch,
# and the GGUF loader falls back to PyTorch SDPA when flash-attn is unavailable.
packages = [
    'gguf', 'gradio', 'transformers', 'sentencepiece',
    'easydict', 'typed-argument-parser', 'seaborn', 'kornia',
    'gputil', 'colorama', 'omegaconf', 'timm==0.9.6',
    'decord', 'pytz', 'imageio', 'einops', 'opencv-python', 'accelerate',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependency installation finished.')


In [ ]:
# Clone the official Python architecture only once.
if not OFFICIAL_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', OFFICIAL_REPO, str(OFFICIAL_DIR)], check=True)
else:
    print('Official Infinity source already exists:', OFFICIAL_DIR)

# Download only the files needed from the unofficial GGUF repository.
from huggingface_hub import hf_hub_download

def download_hf_file(filename, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(
        repo_id=GGUF_REPO,
        filename=filename,
        local_dir=str(target_dir),
    ))

PORT_SCRIPT = download_hf_file('generate_image_2b_q8_gguf.py', PORT_DIR)
PORT_UTILS = download_hf_file('infinity_gguf_utils.py', PORT_DIR)
PATCH_DIR = ROOT / 'gguf_patched_source'
PATCHED_BASIC = download_hf_file('Infinity/infinity/models/basic.py', PATCH_DIR)
PATCHED_INFINITY = download_hf_file('Infinity/infinity/models/infinity.py', PATCH_DIR)

# The GGUF repository includes patched source files with optional attention fallbacks.
# Copy them over the matching files in the official source tree.
official_basic = OFFICIAL_DIR / 'infinity' / 'models' / 'basic.py'
official_infinity = OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py'
shutil.copy2(PATCHED_BASIC, official_basic)
shutil.copy2(PATCHED_INFINITY, official_infinity)

# The patched attention module may intentionally expose flash_attn_func=None.
# Guard the official constructor so it selects the PyTorch SDPA fallback safely.
infinity_source = official_infinity.read_text()
old_attention_guard = "customized_kernel_installed = any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
new_attention_guard = "customized_kernel_installed = flash_attn_func is not None and any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
if old_attention_guard not in infinity_source:
    raise RuntimeError('Expected optional-attention guard was not found in patched infinity.py')
official_infinity.write_text(infinity_source.replace(old_attention_guard, new_attention_guard, 1))
INFINITY_GGUF = download_hf_file('infinity_2b_reg_Q8_0.gguf', ASSET_DIR)
T5_GGUF = download_hf_file('flan-t5-xl-encoder-Q8_0.gguf', ASSET_DIR)
VAE_PATH = download_hf_file('Infinity/infinity_vae_d32_reg.pth', ASSET_DIR)

print('GGUF model:', INFINITY_GGUF)
print('T5 encoder:', T5_GGUF)
print('VAE:', VAE_PATH)
print('Loader:', PORT_SCRIPT)
print('Loader utility:', PORT_UTILS)


In [ ]:
# Verify the expected files before importing the custom loader.
required_files = [PORT_SCRIPT, PORT_UTILS, PATCHED_BASIC, PATCHED_INFINITY, INFINITY_GGUF, T5_GGUF, VAE_PATH]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

for path in required_files:
    print(f'{path.name:40s} {path.stat().st_size / 2**30:.3f} GiB')

assert (OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py').exists(), 'Official Infinity source is incomplete.'
print('All GGUF, VAE, and official source files are present.')


## Memory-efficient T5 loading

The upstream GGUF script first materializes a complete FP16 state dictionary and then creates a second T5 model. That duplicates most of the encoder in host RAM. The replacement below streams one GGUF tensor at a time into an empty Flan-T5-XL model. It preserves the expected 2048-dimensional text features.

A smaller T5 (such as T5-small/base) is not a drop-in replacement: its hidden width and learned feature space differ from the Infinity-2B cross-attention interface. Using one would require a trained projection or distillation step, so this notebook keeps Flan-T5-XL and reduces peak RAM instead.


In [ ]:
import math
import numpy as np
import gguf

def load_t5_encoder_streaming(gguf_path, device='cpu'):
    from gguf import GGUFReader
    from transformers import T5Config, T5EncoderModel

    key_map = {
        'enc.': 'encoder.',
        '.blk.': '.block.',
        'token_embd': 'shared',
        'output_norm': 'final_layer_norm',
        'attn_q': 'layer.0.SelfAttention.q',
        'attn_k': 'layer.0.SelfAttention.k',
        'attn_v': 'layer.0.SelfAttention.v',
        'attn_o': 'layer.0.SelfAttention.o',
        'attn_norm': 'layer.0.layer_norm',
        'attn_rel_b': 'layer.0.SelfAttention.relative_attention_bias',
        'ffn_up': 'layer.1.DenseReluDense.wi_1',
        'ffn_down': 'layer.1.DenseReluDense.wo',
        'ffn_gate': 'layer.1.DenseReluDense.wi_0',
        'ffn_norm': 'layer.1.layer_norm',
    }

    config = T5Config.from_pretrained('google/flan-t5-xl')
    try:
        from accelerate import init_empty_weights
        with init_empty_weights():
            model = T5EncoderModel(config)
        # Materialize directly as FP16 to avoid allocating a full FP32 T5.
        model = model.to(dtype=torch.float16)
        model.to_empty(device=device)
    except Exception as exc:
        raise RuntimeError(
            'Streaming T5 loading requires the accelerate package and empty-weight support. '
            'Restart the runtime and rerun the dependency cell.'
        ) from exc

    model.eval()
    model.requires_grad_(False)
    parameter_refs = dict(model.named_parameters())
    buffer_refs = dict(model.named_buffers())
    reader = GGUFReader(str(gguf_path))
    quantized_types = {gguf.GGMLQuantizationType.F32, gguf.GGMLQuantizationType.F16}
    loaded = 0
    skipped = []

    print(f'[Streaming T5 load] {gguf_path} -> {device}')
    with torch.inference_mode():
        for tensor in reader.tensors:
            name = tensor.name
            for old_key, new_key in key_map.items():
                name = name.replace(old_key, new_key)
            shape = torch.Size(tuple(int(v) for v in reversed(tensor.shape)))
            raw = torch.from_numpy(np.array(tensor.data))
            is_quantized = tensor.tensor_type not in quantized_types
            if is_quantized:
                quant_param = gguf_loader.GGUFParameter(raw, quant_type=tensor.tensor_type)
                value = gguf_loader.dequantize_gguf_tensor(quant_param, target_dtype=torch.float16)
            else:
                value = raw.to(dtype=torch.float16)
            if value.numel() != math.prod(shape):
                skipped.append((name, 'numel mismatch'))
                del raw, value
                continue
            value = value.reshape(shape)
            target = parameter_refs.get(name)
            if target is None:
                target = buffer_refs.get(name)
            if target is None or tuple(target.shape) != tuple(shape):
                skipped.append((name, 'missing or shape mismatch'))
                del raw, value
                continue
            target.data.copy_(value.to(device=target.device, dtype=target.dtype))
            loaded += 1
            del raw, value

    del reader, parameter_refs, buffer_refs
    gc.collect()
    # The model was materialized on the requested device already.
    # Keep this safety path for unusual device-string inputs.
    if str(next(model.parameters()).device) != str(torch.device(device)):
        model.to(device)
    model.eval()
    model.requires_grad_(False)
    print(f'[Streaming T5 load complete] tensors loaded: {loaded}, skipped: {len(skipped)}')
    if skipped:
        print('First skipped tensors:', skipped[:5])
    return model

print('Memory-efficient T5 loader is ready.')


## Import the unofficial loader

The upstream GGUF script contains a NumPy 2 compatibility assignment to `np.ndarray.newbyteorder`. That assignment can fail on some Colab runtimes because NumPy types are immutable. The next cell creates a temporary sanitized copy of the loader and removes only that obsolete compatibility block.


In [ ]:
sys.path.insert(0, str(PORT_DIR))
sys.path.insert(0, str(OFFICIAL_DIR))

loader_source = PORT_SCRIPT.read_text()
compat_pattern = r"\n    # Apply NumPy 2\.0 compatibility patch.*?\n    # Load GGUF state dict"
loader_source, replacements = re.subn(
    compat_pattern,
    '\n    # NumPy compatibility is handled by the installed gguf package.\n    # Load GGUF state dict',
    loader_source,
    count=1,
    flags=re.S,
)
print('Removed obsolete NumPy compatibility block:', replacements == 1)

PATCHED_LOADER = PORT_DIR / 'generate_image_2b_q8_gguf_colab.py'
PATCHED_LOADER.write_text(loader_source)
spec = importlib.util.spec_from_file_location('infinity_gguf_colab_loader', PATCHED_LOADER)
gguf_loader = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = gguf_loader
spec.loader.exec_module(gguf_loader)
print('Custom GGUF loader imported successfully.')


## Load all components

The T5 encoder is streamed directly to CUDA by default. The VAE and quantized Infinity transformer are placed on the GPU.


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('A CUDA GPU is required for practical inference. Select a GPU runtime and rerun.')

print('[1/4] Loading T5 tokenizer...')
text_tokenizer = gguf_loader.load_t5_tokenizer_from_gguf(str(T5_GGUF))

print(f'[2/4] Streaming quantized T5 encoder to {T5_DEVICE}...')
text_encoder = load_t5_encoder_streaming(str(T5_GGUF), device=T5_DEVICE)

print('[3/4] Loading VAE on GPU...')
vae = gguf_loader.load_vae(str(VAE_PATH), vae_type=32, device=DEVICE)

print('[4/4] Loading quantized Infinity-2B transformer on GPU...')
infinity_model = gguf_loader.load_infinity_from_gguf(
    str(INFINITY_GGUF),
    vae=vae,
    device=DEVICE,
    model_type='infinity_2b',
    text_channels=2048,
    pn=MODEL_PN,
)

infinity_model.eval()
vae.eval()
print('All components loaded successfully.')


In [ ]:
# Build the official dynamic-resolution schedule for the selected preset.
import numpy as np
from infinity.utils.dynamic_resolution import dynamic_resolution_h_w, h_div_w_templates

ASPECT_RATIO = 1.0
h_div_w_template = h_div_w_templates[np.argmin(np.abs(h_div_w_templates - ASPECT_RATIO))]
scale_schedule = dynamic_resolution_h_w[h_div_w_template][MODEL_PN]['scales']
scale_schedule = [(1, h, w) for (_, h, w) in scale_schedule]
print('Aspect ratio:', h_div_w_template)
print('Preset:', MODEL_PN)
print('Scale schedule:', scale_schedule)


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def tensor_to_pil(image):
    """Convert common Infinity output layouts/ranges into an RGB PIL image."""
    if isinstance(image, (list, tuple)):
        image = image[0]
    tensor = image.detach().float().cpu() if torch.is_tensor(image) else torch.as_tensor(image).float()
    if tensor.ndim == 4:
        tensor = tensor[0]
    if tensor.ndim != 3:
        raise ValueError(f'Unexpected image shape: {tuple(tensor.shape)}')
    if tensor.shape[0] in (1, 3, 4):
        tensor = tensor.permute(1, 2, 0)
    if tensor.shape[-1] == 1:
        tensor = tensor.repeat(1, 1, 3)
    if tensor.shape[-1] > 3:
        tensor = tensor[..., :3]
    lo, hi = float(tensor.min()), float(tensor.max())
    if lo < -0.05:
        tensor = (tensor + 1.0) / 2.0
    elif hi > 1.05:
        tensor = tensor / 255.0
    array = (tensor.clamp(0, 1).numpy() * 255).round().astype('uint8')
    return Image.fromarray(array, mode='RGB')

def generate_one(prompt, seed=SEED, output_path=None):
    started = time.time()
    with torch.inference_mode():
        image = gguf_loader.generate_image(
            infinity_model, vae, text_tokenizer, text_encoder, prompt,
            cfg_scale=CFG_SCALE,
            tau=TAU,
            seed=seed,
            scale_schedule=scale_schedule,
            vae_type=32,
            device=DEVICE,
        )
    pil = tensor_to_pil(image)
    if output_path is not None:
        pil.save(output_path)
    del image
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Generated in {time.time() - started:.2f}s:', prompt)
    return pil


In [ ]:
import types
from contextlib import nullcontext
import torchvision
import torch.nn.functional as F
from tqdm.auto import tqdm
from PIL import Image, ImageOps
from infinity.models.basic import CrossAttnBlock, apply_rotary_emb, slow_attn
from infinity.models.infinity import sample_with_top_k_top_p_also_inplace_modifying_logits_

RUNTIME_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
device = DEVICE
infinity = infinity_model
SCALE_SCHEDULE = scale_schedule
PATCH_NUMS = tuple(h for (_, h, w) in SCALE_SCHEDULE)
IMAGE_SIZE_HW = (1024, 1024) if MODEL_PN == '1M' else (512, 512)

if not hasattr(vae.quantizer, 'lfq'):
    vae.quantizer.lfq = vae.quantizer.bsq

print('Notebook 18 compatibility aliases ready.')
print('Backend: Infinity-2B GGUF | device:', device, '| image size:', IMAGE_SIZE_HW)


## CSD100 20-Pair Configuration

These are fixed CSD100 pairs for comparing the three PFB + SAC variants. The earlier hard `horseshoe+graffiti` content case is replaced by the simpler `bottle+drawing` item. PFB + SAC uses a slightly more specific text prompt, `a photo of <1-2 word descriptor> <content object>`; the real content image is displayed only as a reference.


In [ ]:
from dataclasses import dataclass
import csv
import itertools
import json
import pandas as pd

WORKSPACE_REPO = 'https://github.com/LeeHoang2710/Style-Transfer-Experiment.git'
WORKSPACE = Path('/Users/builehoang/Documents/Projects/Image Generation/VAR_Style_Transfer_Workspace')
if not WORKSPACE.exists():
    WORKSPACE = Path('/content/VAR_Style_Transfer_Workspace')
    if not WORKSPACE.exists() and Path('/content').exists():
        subprocess.run(['git', 'clone', '--depth', '1', WORKSPACE_REPO, str(WORKSPACE)], check=True)
    if not WORKSPACE.exists():
        raise FileNotFoundError('Could not find VAR_Style_Transfer_Workspace locally or under /content.')

CSD100_DIR = WORKSPACE / 'csd100'
if not CSD100_DIR.exists():
    raise FileNotFoundError(f'Missing CSD100 directory: {CSD100_DIR}')

KINGFISHER_OUTPUT_DIR = WORKSPACE / 'outputs' / 'notebook_13_kingfisher_csd100'
KINGFISHER_PAIR_OUTPUT_DIR = KINGFISHER_OUTPUT_DIR / 'pairs'

OUTPUT_DIR = WORKSPACE / 'outputs' / 'notebook_18_diagnostic_selected_object_masked_pfb_sac_seed2026'
BASELINE_DIR = OUTPUT_DIR / 'baseline_content_stream'
MASK_DIR = OUTPUT_DIR / 'content_object_masks'
MASK_OVERLAY_DIR = OUTPUT_DIR / 'content_object_mask_overlays'
STYLE_MASK_DIR = OUTPUT_DIR / 'style_reference_object_masks'
STYLE_MASK_OVERLAY_DIR = OUTPUT_DIR / 'style_reference_object_mask_overlays'
IMPROVED_DIR = OUTPUT_DIR / 'diagnostic_selected_object_masked_pfb_sac'
SCALE_SELECTION_DIR = OUTPUT_DIR / 'scale_selection_diagnostic'
TRACE_DIR = OUTPUT_DIR / 'traces'
DIAGNOSTIC_DIR = OUTPUT_DIR / 'scale_diagnostics'
GALLERY_DIR = OUTPUT_DIR / 'galleries'
EVAL_DIR = OUTPUT_DIR / 'evaluation'

for path in [
    BASELINE_DIR, MASK_DIR, MASK_OVERLAY_DIR, STYLE_MASK_DIR, STYLE_MASK_OVERLAY_DIR,
    IMPROVED_DIR, SCALE_SELECTION_DIR, TRACE_DIR, DIAGNOSTIC_DIR, GALLERY_DIR, EVAL_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)


def parse_csd100_item(folder):
    folder = Path(folder)
    if '+' not in folder.name:
        raise ValueError(f'CSD100 folder name must contain +: {folder.name}')
    object_label, style_label = folder.name.split('+', 1)
    clean_object = object_label.replace('_', ' ').replace('-', ' ')
    clean_style = style_label.replace('_', ' ').replace('-', ' ')
    image_path = folder / '00.jpg'
    if not image_path.exists():
        raise FileNotFoundError(image_path)
    return {
        'folder': folder,
        'image_path': image_path,
        'object_label': clean_object,
        'style_label': clean_style,
        'item_id': folder.name,
    }

csd_items = [parse_csd100_item(path) for path in sorted(CSD100_DIR.iterdir()) if path.is_dir() and (path / '00.jpg').exists()]
assert len(csd_items) >= 40, f'Expected at least 40 CSD100 items, found {len(csd_items)}'

EXAMPLE_PAIRS = [
    ('fox+graffiti', 'pen+artwork'),
    ('mushroom+melting_golden_3D_rendering', 'horse+rainbow_flowing_smoke_wave'),
    ('scarecrow+melting_golden_3D_rendering', 'cat+glowing'),
    ('bear+glowing', 'teapot+psychedelic'),
    ('piano+impressionism', 'duck+blueprint'),
    ('duck+blueprint', 'flower+mosaic'),
    ('flower+pixel', 'brush+watercolor'),
    ('moose+origami', 'cat+glowing'),
    ('bottle+drawing', 'lantern+line_drawing_illustration_art'),
    ('muffin+drawing', 'teacup+comic'),
    ('camera+artwork', 'leopard+geometric'),
    ('camera+origami', 'lollipop+origami'),
    ('car+rainbow_flowing_smoke_wave', 'microphone+minimal_pastel_colors_art'),
    ('balloon+mosaic', 'crown+art'),
    ('bicycle+blueprint', 'saxophone+pop'),
    ('glass+watercolor_and_ink_wash', 'robot+woodcut'),
    ('compass+flat_cartoon_illustration_art', 'watermelon+papercut'),
    ('rabbit+sticker', 'notebook+melting_golden_3D_rendering'),
    ('turtle+origami', 'fox+minimal_pastel_colors_art'),
    ('umbrella+melting_golden_3D_rendering', 'seagull+geometric'),
]

items_by_id = {item['item_id']: item for item in csd_items}
missing_pair_items = sorted({item_id for pair in EXAMPLE_PAIRS for item_id in pair} - set(items_by_id))
if missing_pair_items:
    raise FileNotFoundError('Missing CSD100 example item folders: ' + ', '.join(missing_pair_items))

OBJECT_PROMPT_DESCRIPTORS = {
    'fox': 'red fox',
    'mushroom': 'golden mushroom',
    'scarecrow': 'straw scarecrow',
    'bear': 'cute bear',
    'piano': 'grand piano',
    'duck': 'rubber duck',
    'flower': 'pink flower',
    'moose': 'paper moose',
    'bottle': 'glass bottle',
    'muffin': 'frosted muffin',
    'camera': 'vintage camera',
    'car': 'sports car',
    'balloon': 'hot air balloon',
    'bicycle': 'blue bicycle',
    'glass': 'clear glass',
    'compass': 'brass compass',
    'rabbit': 'white rabbit',
    'turtle': 'green turtle',
    'umbrella': 'red umbrella',
}


def detailed_object_phrase(object_label):
    return OBJECT_PROMPT_DESCRIPTORS.get(str(object_label), str(object_label))

PAIR_ROWS = []
for pair_id, (content_item_id, style_item_id) in enumerate(EXAMPLE_PAIRS):
    content_item = items_by_id[content_item_id]
    style_item = items_by_id[style_item_id]
    pfb_prompt = f"a photo of {detailed_object_phrase(content_item['object_label'])}"
    kingfisher_prompt = f"a photo of {content_item['object_label']} in {style_item['style_label']} style"
    PAIR_ROWS.append({
        'pair_id': pair_id,
        'content_id': content_item['item_id'],
        'content_object': content_item['object_label'],
        'content_style': content_item['style_label'],
        'content_path': content_item['image_path'],
        'style_id': style_item['item_id'],
        'style_object': style_item['object_label'],
        'style_label': style_item['style_label'],
        'style_path': style_item['image_path'],
        'pfb_prompt': pfb_prompt,
        'kingfisher_prompt': kingfisher_prompt,
    })

pair_table = pd.DataFrame([{k: str(v) for k, v in row.items()} for row in PAIR_ROWS])
display(pair_table[['pair_id', 'content_id', 'content_object', 'content_style', 'style_id', 'style_object', 'style_label', 'pfb_prompt']])
print('CSD100 items:', len(csd_items))
print('Selected pairs:', len(PAIR_ROWS))
print('Output directory:', OUTPUT_DIR)


## Experiment Configuration


In [ ]:
CFG = CFG_SCALE
TOP_K = 600
TOP_P = 0.95
PAPER_ALPHA = 1.0
SAC_START = 2

# Fixed global-style branch from the current best scale-aware hypothesis.
GLOBAL_STYLE_STEPS = [0, 1, 2]
GLOBAL_SVD_RANK = 1
GLOBAL_STYLE_STRENGTH_BY_STEP = {
    0: 1.0,
    1: 0.75,
    2: 0.5625,
}

# Candidate object-specific scales are selected by the diagnostic, not manually.
CANDIDATE_OBJECT_STEPS = [step for step in range(3, len(SCALE_SCHEDULE))]
SELECT_TOP_K_OBJECT_STEPS = 3
OBJECT_FOREGROUND_RANK = 2
OBJECT_BACKGROUND_RANK = 1
OBJECT_FOREGROUND_STRENGTH = 1.25
OBJECT_FOREGROUND_DECAY = 0.75
MASKED_BACKGROUND_STRENGTH = 0.15

# Soft-mask settings. Dilation expands the object slightly; blur avoids hard boundaries.
SOFT_MASK_DILATION_SIZE = 15  # odd integer for PIL MaxFilter
SOFT_MASK_BLUR_RADIUS = 8

# Scale-selection score weights.
OBJECT_SCORE_BACKGROUND_WEIGHT = 0.5
OBJECT_SCORE_LEAKAGE_WEIGHT = 1.0
SCALE_SELECTION_NUM_PAIRS = 50

VARIANT_ORDER = ['diagnostic_selected_object_masked']
VARIANT_LABELS = {
    'diagnostic_selected_object_masked': 'diagnostic top-k object-mask\nPFB+SAC',
}
VARIANT_INJECTED_INDICES = {
    'diagnostic_selected_object_masked': [],  # filled after scale selection
}
SELECTED_OBJECT_STEPS = []
SELECTED_FEATURE_INDICES = []
SELECTED_STYLE_STRENGTH_BY_STEP = {}

CLIPSEG_MODEL_ID = 'CIDAS/clipseg-rd64-refined'
CLIPSEG_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CONTENT_MASK_THRESHOLD = None
STYLE_MASK_PROMPT = 'main object'
STYLE_MASK_THRESHOLD = None

if max(GLOBAL_STYLE_STEPS) >= len(SCALE_SCHEDULE):
    raise RuntimeError(f'This notebook needs at least {max(GLOBAL_STYLE_STEPS) + 1} Infinity scales; got {len(SCALE_SCHEDULE)}.')

print('Global style steps:', GLOBAL_STYLE_STEPS, '| global rank:', GLOBAL_SVD_RANK)
print('Candidate object steps:', CANDIDATE_OBJECT_STEPS, '| selected top-k:', SELECT_TOP_K_OBJECT_STEPS)
print('Object foreground rank:', OBJECT_FOREGROUND_RANK, '| background strength:', MASKED_BACKGROUND_STRENGTH)
print('Soft mask dilation:', SOFT_MASK_DILATION_SIZE, '| blur:', SOFT_MASK_BLUR_RADIUS)
print('Score weights: bg=', OBJECT_SCORE_BACKGROUND_WEIGHT, '| leakage=', OBJECT_SCORE_LEAKAGE_WEIGHT)
print('CLIPSeg:', CLIPSEG_MODEL_ID, '| device:', CLIPSEG_DEVICE)


## PFB And SAC Helpers


In [ ]:

def load_reference_image(path, size=512):
    image = Image.open(path).convert('RGB')
    image = ImageOps.fit(image, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    image_01 = torchvision.transforms.functional.to_tensor(image).unsqueeze(0).to(device)
    return image_01.mul(2).sub(1), image_01, image


@torch.no_grad()
def extract_multiscale_style_features(image_m11):
    # Infinity's equivalent of the paper's F_s: cumulative BSQ residual codes
    # at the final latent resolution, one snapshot after each AR scale.
    with torch.amp.autocast('cuda', enabled=False):
        _, _, _, all_bit_indices, _, _ = vae.encode(image_m11.float(), scale_schedule=SCALE_SCHEDULE)

    summed_codes = None
    features = []
    final_size = SCALE_SCHEDULE[-1]
    num_scales = len(SCALE_SCHEDULE)
    for step_id, bit_indices in enumerate(all_bit_indices):
        codes = vae.quantizer.lfq.indices_to_codes(bit_indices, label_type='bit_label')
        if step_id != num_scales - 1:
            codes = F.interpolate(codes, size=final_size, mode=vae.quantizer.z_interplote_up)
        summed_codes = codes if summed_codes is None else summed_codes + codes
        features.append(summed_codes.detach().float().clone())
    assert len(features) == len(PATCH_NUMS)
    return features


def phi_svd(feature, alpha=1.0, rank=None):
    '''Paper Eq. (5), generalized from VAR [B,C,H,W] to Infinity [B,C,T,H,W].'''
    original_dtype = feature.dtype
    batch, channels = feature.shape[:2]
    spatial_shape = feature.shape[2:]
    outputs = []

    for batch_id in range(batch):
        matrix = feature[batch_id].detach().float().reshape(channels, -1)
        u, singular_values, vh = torch.linalg.svd(matrix, full_matrices=False)
        available_rank = singular_values.numel()
        used_rank = available_rank if rank is None else min(int(rank), available_rank)
        weights = torch.exp(
            -float(alpha) * torch.arange(used_rank, device=matrix.device, dtype=matrix.dtype)
        )
        weighted_s = singular_values[:used_rank] * weights
        reconstructed = (u[:, :used_rank] * weighted_s.unsqueeze(0)) @ vh[:used_rank]
        outputs.append(reconstructed.reshape(channels, *spatial_shape))

    return torch.stack(outputs).to(dtype=original_dtype)


def principal_feature_blend(generation_feature, style_feature, alpha=1.0, rank=None, strength=1.0):
    '''PFB with optional strength multiplier; strength=1 is the paper method.'''
    if generation_feature.shape != style_feature.shape:
        raise ValueError(f'PFB shape mismatch: {generation_feature.shape} vs {style_feature.shape}')
    style_feature = style_feature.to(generation_feature)
    style_component = phi_svd(style_feature, alpha=alpha, rank=rank)
    generation_component = phi_svd(generation_feature, alpha=alpha, rank=rank)
    return generation_feature + float(strength) * (style_component - generation_component)


def apply_feature_edit(generation_feature, style_feature, mode, alpha=1.0, rank=None, strength=1.0):
    if mode == 'none':
        return generation_feature
    if mode == 'replace':
        return style_feature.to(generation_feature)
    if mode == 'pfb':
        return principal_feature_blend(
            generation_feature, style_feature, alpha=alpha, rank=rank, strength=strength
        )
    raise ValueError(f'Unknown edit mode: {mode}')


# --- Object-masked PFB extensions ---

def resize_mask_to_feature(mask, feature):
    if mask is None:
        return None
    mask = mask.detach().float().to(device=feature.device)
    while mask.ndim > 2 and mask.shape[0] == 1:
        mask = mask.squeeze(0)
    if mask.ndim != 2:
        raise ValueError(f'Expected a 2D mask after squeezing singleton dimensions, got {tuple(mask.shape)}')
    mask = mask.unsqueeze(0).unsqueeze(0)
    mask = F.interpolate(mask, size=feature.shape[-2:], mode='bilinear', align_corners=False)
    if feature.ndim == 5:
        mask = mask.unsqueeze(2)
    while mask.ndim < feature.ndim:
        mask = mask.unsqueeze(0)
    return mask.to(dtype=feature.dtype).clamp(0, 1)


def principal_feature_blend_regions(
    generation_feature,
    style_feature,
    *,
    target_mask,
    style_mask,
    alpha=1.0,
    foreground_rank=1,
    background_rank=1,
    foreground_strength=1.0,
    background_strength=0.0,
):
    if generation_feature.shape != style_feature.shape:
        raise ValueError(f'PFB shape mismatch: {generation_feature.shape} vs {style_feature.shape}')
    style_feature = style_feature.to(generation_feature)
    target_foreground = resize_mask_to_feature(target_mask, generation_feature)
    target_background = 1.0 - target_foreground

    style_foreground_mask = resize_mask_to_feature(style_mask, style_feature) if style_mask is not None else None
    if style_foreground_mask is None:
        style_foreground = style_feature
        style_background = style_feature
    else:
        style_foreground = style_feature * style_foreground_mask
        style_background = style_feature * (1.0 - style_foreground_mask)

    generation_foreground = generation_feature * target_foreground
    generation_background = generation_feature * target_background

    foreground_delta = phi_svd(style_foreground, alpha=alpha, rank=foreground_rank) - phi_svd(
        generation_foreground, alpha=alpha, rank=foreground_rank
    )
    background_delta = phi_svd(style_background, alpha=alpha, rank=background_rank) - phi_svd(
        generation_background, alpha=alpha, rank=background_rank
    )

    return (
        generation_feature
        + float(foreground_strength) * target_foreground * foreground_delta
        + float(background_strength) * target_background * background_delta
    )


def apply_feature_edit(
    generation_feature,
    style_feature,
    mode,
    alpha=1.0,
    rank=None,
    strength=1.0,
    mask=None,
    style_mask=None,
    split_style_regions=False,
    background_strength=0.0,
    foreground_rank=None,
    background_rank=None,
):
    if mode == 'none':
        return generation_feature
    if mode == 'replace':
        return style_feature.to(generation_feature)
    if mode == 'pfb':
        if split_style_regions and mask is not None:
            return principal_feature_blend_regions(
                generation_feature,
                style_feature,
                target_mask=mask,
                style_mask=style_mask,
                alpha=alpha,
                foreground_rank=rank if foreground_rank is None else foreground_rank,
                background_rank=rank if background_rank is None else background_rank,
                foreground_strength=strength,
                background_strength=background_strength,
            )
        return principal_feature_blend(
            generation_feature,
            style_feature,
            alpha=alpha,
            rank=rank,
            strength=strength,
        )
    raise ValueError(f'Unknown edit mode: {mode}')


In [ ]:

class SACController:
    def __init__(self, base_batch=1):
        self.base_batch = base_batch
        self.active = False
        self.sac_strength = 1.0
        self.total_calls = 0
        self.max_q_copy_error = 0.0
        self.max_k_copy_error = 0.0

    def reset_statistics(self):
        self.total_calls = 0
        self.max_q_copy_error = 0.0
        self.max_k_copy_error = 0.0


def _infinity_sac_attention_forward(attention, x, attn_bias_or_two_vector, attn_fn=None, scale_schedule=None, rope2d_freqs_grid=None, scale_ind=0):
    batch4, length, channels = x.shape
    if attention.using_flash:
        raise RuntimeError('Notebook SAC patch expects customized_flash_attn=False.')

    # GGUF replaces mat_qkv with GGUFLinear; its forward dequantizes Byte weights on demand.
    qkv = attention.mat_qkv(x)
    qkv = qkv + torch.cat((attention.q_bias, attention.zero_k_bias, attention.v_bias)).to(qkv)
    qkv = qkv.view(batch4, length, 3, attention.num_heads, attention.head_dim)
    q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(dim=0)

    if attention.cos_attn:
        scale_mul = attention.scale_mul_1H11.clamp_max(attention.max_scale_mul).exp()
        q = F.normalize(q, dim=-1, eps=1e-12).mul(scale_mul).contiguous()
        k = F.normalize(k, dim=-1, eps=1e-12).contiguous()
        v = v.contiguous()
    else:
        q, k, v = q.contiguous(), k.contiguous(), v.contiguous()

    if rope2d_freqs_grid is not None:
        q, k = apply_rotary_emb(
            q, k, scale_schedule, rope2d_freqs_grid,
            attention.pad_to_multiplier, attention.rope2d_normalized_by_hw, scale_ind,
        )

    controller = getattr(attention, '_paper_sac_controller', None)
    if controller is not None and controller.active:
        b = controller.base_batch
        if batch4 != 4 * b:
            raise RuntimeError(f'SAC expected joint batch {4 * b}, received {batch4}.')

        # Order: content conditional, generation conditional,
        #        content unconditional, generation unconditional.
        q_content = torch.cat((q[:b], q[:b], q[2*b:3*b], q[2*b:3*b]), dim=0)
        k_content = torch.cat((k[:b], k[:b], k[2*b:3*b], k[2*b:3*b]), dim=0)
        if controller.sac_strength >= 1.0:
            q, k = q_content, k_content
        else:
            q = q + float(controller.sac_strength) * (q_content - q)
            k = k + float(controller.sac_strength) * (k_content - k)

        controller.total_calls += 1
        controller.max_q_copy_error = max(
            controller.max_q_copy_error,
            float((q[b:2*b] - q[:b]).abs().max().detach().cpu()),
            float((q[3*b:4*b] - q[2*b:3*b]).abs().max().detach().cpu()),
        )
        controller.max_k_copy_error = max(
            controller.max_k_copy_error,
            float((k[b:2*b] - k[:b]).abs().max().detach().cpu()),
            float((k[3*b:4*b] - k[2*b:3*b]).abs().max().detach().cpu()),
        )

    if attention.caching:
        if attention.cached_k is None:
            attention.cached_k, attention.cached_v = k, v
        else:
            attention.cached_k = torch.cat((attention.cached_k, k), dim=2)
            attention.cached_v = torch.cat((attention.cached_v, v), dim=2)
        k, v = attention.cached_k, attention.cached_v

    if attention.use_flex_attn and attn_fn is not None:
        output = attn_fn(q, k, v, scale=attention.scale).transpose(1, 2).reshape(batch4, length, channels)
    else:
        output = slow_attn(
            query=q.to(v.dtype), key=k.to(v.dtype), value=v,
            scale=attention.scale, attn_mask=attn_bias_or_two_vector, dropout_p=0,
        ).transpose(1, 2).reshape(batch4, length, channels)
    return attention.proj_drop(attention.proj(output))


class PaperSACPatch:
    def __init__(self, model, controller):
        self.model = model
        self.controller = controller
        self.original_forwards = []

    def __enter__(self):
        for block in self.model.unregistered_blocks:
            if not isinstance(block, CrossAttnBlock):
                continue
            attention = block.sa
            self.original_forwards.append((attention, attention.forward))
            attention._paper_sac_controller = self.controller
            attention.forward = types.MethodType(_infinity_sac_attention_forward, attention)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        for attention, original_forward in self.original_forwards:
            attention.forward = original_forward
            if hasattr(attention, '_paper_sac_controller'):
                delattr(attention, '_paper_sac_controller')
        return False


In [ ]:

def encode_prompts(prompts, enable_positive_prompt=False):
    if isinstance(prompts, str):
        prompts = [prompts]
    tokens = text_tokenizer(
        text=list(prompts), max_length=512, padding='max_length', truncation=True, return_tensors='pt'
    )
    input_ids = tokens.input_ids.to(device, non_blocking=True)
    mask = tokens.attention_mask.to(device, non_blocking=True)
    with torch.no_grad():
        text_features = text_encoder(input_ids=input_ids, attention_mask=mask)['last_hidden_state'].float()
    lens = mask.sum(dim=-1).tolist()
    cu_seqlens_k = F.pad(mask.sum(dim=-1).to(dtype=torch.int32).cumsum_(0), (1, 0))
    max_seqlen_k = max(lens)
    kv_compact = []
    for len_i, feat_i in zip(lens, text_features.unbind(0)):
        kv_compact.append(feat_i[:len_i])
    kv_compact = torch.cat(kv_compact, dim=0)
    return kv_compact, lens, cu_seqlens_k, max_seqlen_k


def _sample_bit_labels(logits_bl2d, rng, top_k, top_p):
    batch, seq_len = logits_bl2d.shape[:2]
    logits = logits_bl2d.reshape(batch, -1, 2).clone()
    sampled = sample_with_top_k_top_p_also_inplace_modifying_logits_(
        logits, rng=rng, top_k=top_k, top_p=top_p, num_samples=1
    )[:, :, 0]
    return sampled.reshape(batch, seq_len, -1)


def _bit_labels_to_codes(idx_bld, pn):
    idx = idx_bld.reshape(idx_bld.shape[0], pn[1], pn[2], -1)
    idx = idx.unsqueeze(1)  # [B, 1, h, w, d]
    return vae.quantizer.lfq.indices_to_codes(idx, label_type='bit_label')


def _next_raw_from_summed_codes(summed_codes, next_scale):
    last_stage = F.interpolate(summed_codes, size=next_scale, mode=vae.quantizer.z_interplote_up)
    last_stage = last_stage.squeeze(-3)
    if infinity.apply_spatial_patchify:
        last_stage = torch.nn.functional.pixel_unshuffle(last_stage, 2)
    last_stage = last_stage.reshape(*last_stage.shape[:2], -1).permute(0, 2, 1)
    return last_stage


def _decode_summed_codes_to_image_01(summed_codes):
    image = vae.decode(summed_codes.squeeze(-3))
    return image.add(1).mul(0.5).clamp(0, 1)


@torch.no_grad()
def paper_dual_path_generate(
    model,
    prompt,
    style_features,
    *,
    seed=42,
    cfg=1.0,
    tau=0.1,
    top_k=900,
    top_p=0.97,
    pfb_feature_index=2,
    pfb_feature_indices=None,
    sac_prediction_start=3,
    edit_mode='pfb',
    alpha=1.0,
    rank=None,
    style_strength=1.0,
    style_decay=1.0,
    style_strength_by_step=None,
    feature_masks_by_step=None,
    style_masks_by_step=None,
    masked_background_strength=0.0,
    split_style_regions=False,
    foreground_rank=None,
    background_rank=None,
    sac_strength=1.0,
    enable_sac=True,
):
    '''Paper Algorithm 1 adapted to Infinity's bitwise 0.25M GGUF inference.

    The paper writes SAC as active from the fine stage s=3. Here the code uses
    zero-based stage indices, so SAC starts at index 2 and remains active for
    every later stage. The multi-scale PFB configuration is an experiment
    added on top of the paper's single PFB intervention at F3.
    '''
    if cfg < 1.0:
        raise ValueError('CFG must be >= 1.0 for this dual-stream experiment.')
    if pfb_feature_indices is None:
        pfb_feature_indices = [pfb_feature_index]
    else:
        pfb_feature_indices = sorted(set(int(index) for index in pfb_feature_indices))
    if not pfb_feature_indices or any(index < 0 or index >= len(SCALE_SCHEDULE) for index in pfb_feature_indices):
        raise ValueError('Invalid PFB feature indices.')
    if style_strength_by_step is not None:
        style_strength_by_step = {int(step): float(strength) for step, strength in style_strength_by_step.items()}
        expected_steps = set(pfb_feature_indices)
        if set(style_strength_by_step) != expected_steps or any(strength < 0 for strength in style_strength_by_step.values()):
            raise ValueError('style_strength_by_step must provide one non-negative strength for every PFB scale.')
    if feature_masks_by_step is None:
        feature_masks_by_step = {}
    else:
        feature_masks_by_step = {int(step): mask for step, mask in feature_masks_by_step.items()}
    if style_masks_by_step is None:
        style_masks_by_step = {}
    else:
        style_masks_by_step = {int(step): mask for step, mask in style_masks_by_step.items()}
    if enable_sac and not 0 <= sac_prediction_start < len(SCALE_SCHEDULE):
        raise ValueError('Invalid SAC prediction start.')

    model.eval()
    base_batch = 1
    condition_batch = 2  # content stream + generation stream
    content_rng = torch.Generator(device=device).manual_seed(seed)
    generation_rng = torch.Generator(device=device).manual_seed(seed)

    kv_compact, lens, cu_seqlens_k, max_seqlen_k = encode_prompts([prompt, prompt])
    kv_compact_un = kv_compact.clone()
    total = 0
    for le in lens:
        kv_compact_un[total:total + le] = model.cfg_uncond[:le]
        total += le
    kv_compact = torch.cat((kv_compact, kv_compact_un), dim=0)
    cu_seqlens_k = torch.cat((cu_seqlens_k, cu_seqlens_k[1:] + cu_seqlens_k[-1]), dim=0)
    bs = 4

    kv_compact = model.text_norm(kv_compact)
    sos = cond_BD = model.text_proj_for_sos((kv_compact, cu_seqlens_k, max_seqlen_k))
    kv_compact = model.text_proj_for_ca(kv_compact)
    ca_kv = kv_compact, cu_seqlens_k, max_seqlen_k
    last_stage = sos.unsqueeze(1).expand(bs, 1, -1) + model.pos_start.expand(bs, 1, -1)

    with torch.amp.autocast('cuda', enabled=False):
        cond_BD_or_gss = model.shared_ada_lin(cond_BD.float()).float().contiguous()

    final_size = SCALE_SCHEDULE[-1]
    content_summed = last_stage.new_zeros(base_batch, model.d_vae, *final_size)
    generation_summed = torch.zeros_like(content_summed)
    content_trace, generation_trace = [], []

    controller = SACController(base_batch)
    controller.sac_strength = float(sac_strength)
    controller.reset_statistics()

    for block in model.unregistered_blocks:
        block.sa.kv_caching(True)

    pre_pfb_max_difference = 0.0
    pfb_relative_change_by_step = {}
    try:
        with torch.amp.autocast('cuda', enabled=True, dtype=torch.bfloat16, cache_enabled=True):
            sac_context = PaperSACPatch(model, controller) if enable_sac else nullcontext()
            with sac_context:
                for step_id, pn in enumerate(SCALE_SCHEDULE):
                    controller.active = enable_sac and step_id >= sac_prediction_start
                    need_to_pad = 0
                    attn_fn = None
                    if model.use_flex_attn:
                        attn_fn = model.attn_fn_compile_dict.get(tuple(SCALE_SCHEDULE[:step_id + 1]), None)

                    layer_idx = 0
                    for block_idx, block_chunk in enumerate(model.block_chunks):
                        if model.add_lvl_embeding_only_first_block and block_idx == 0:
                            last_stage = model.add_lvl_embeding(last_stage, step_id, SCALE_SCHEDULE, need_to_pad=need_to_pad)
                        if not model.add_lvl_embeding_only_first_block:
                            last_stage = model.add_lvl_embeding(last_stage, step_id, SCALE_SCHEDULE, need_to_pad=need_to_pad)

                        for block in block_chunk.module:
                            last_stage = block(
                                x=last_stage,
                                cond_BD=cond_BD_or_gss,
                                ca_kv=ca_kv,
                                attn_bias_or_two_vector=None,
                                attn_fn=attn_fn,
                                scale_schedule=SCALE_SCHEDULE,
                                rope2d_freqs_grid=model.rope2d_freqs_grid,
                                scale_ind=step_id,
                            )
                            layer_idx += 1

                    logits = model.get_logits(last_stage, cond_BD).mul(1 / float(tau))
                    logits = float(cfg) * logits[:condition_batch] + (1 - float(cfg)) * logits[condition_batch:]
                    content_idx = _sample_bit_labels(logits[:1], content_rng, top_k, top_p)
                    generation_idx = _sample_bit_labels(logits[1:2], generation_rng, top_k, top_p)

                    content_codes = _bit_labels_to_codes(content_idx, pn)
                    generation_codes = _bit_labels_to_codes(generation_idx, pn)
                    if step_id != len(SCALE_SCHEDULE) - 1:
                        content_codes = F.interpolate(content_codes, size=final_size, mode=vae.quantizer.z_interplote_up)
                        generation_codes = F.interpolate(generation_codes, size=final_size, mode=vae.quantizer.z_interplote_up)

                    content_summed = content_summed + content_codes
                    generation_summed = generation_summed + generation_codes

                    if step_id < min(pfb_feature_indices):
                        pre_pfb_max_difference = max(
                            pre_pfb_max_difference,
                            float((content_summed - generation_summed).abs().max().detach().cpu()),
                        )

                    if step_id in pfb_feature_indices and edit_mode != 'none':
                        injection_order = pfb_feature_indices.index(step_id)
                        effective_strength = (
                            style_strength_by_step[step_id]
                            if style_strength_by_step is not None
                            else float(style_strength) * float(style_decay) ** injection_order
                        )
                        generation_before_edit = generation_summed.clone()
                        generation_summed = apply_feature_edit(
                            generation_summed,
                            style_features[step_id],
                            mode=edit_mode,
                            alpha=alpha,
                            rank=rank,
                            strength=effective_strength,
                            mask=feature_masks_by_step.get(step_id),
                            style_mask=style_masks_by_step.get(step_id),
                            split_style_regions=split_style_regions,
                            background_strength=masked_background_strength,
                            foreground_rank=foreground_rank,
                            background_rank=background_rank,
                        )
                        pfb_relative_change_by_step[step_id] = float(
                            (generation_summed - generation_before_edit).norm()
                            / generation_before_edit.norm().clamp_min(1e-8)
                        )

                    content_trace.append(content_summed.detach().float().clone())
                    generation_trace.append(generation_summed.detach().float().clone())

                    if step_id != len(SCALE_SCHEDULE) - 1:
                        next_scale = SCALE_SCHEDULE[step_id + 1]
                        content_next = _next_raw_from_summed_codes(content_summed, next_scale)
                        generation_next = _next_raw_from_summed_codes(generation_summed, next_scale)
                        two_streams = torch.cat((content_next, generation_next), dim=0)
                        last_stage = model.word_embed(model.norm0_ve(two_streams))
                        last_stage = last_stage.repeat(bs // condition_batch, 1, 1)

        content_image = _decode_summed_codes_to_image_01(content_summed)
        generation_image = _decode_summed_codes_to_image_01(generation_summed)
        return {
            'content_image_01': content_image,
            'stylized_image_01': generation_image,
            'content_features': content_trace,
            'generation_features': generation_trace,
            'pre_pfb_max_difference': pre_pfb_max_difference,
            'sac_calls': controller.total_calls,
            'max_q_copy_error': controller.max_q_copy_error,
            'max_k_copy_error': controller.max_k_copy_error,
            'pfb_relative_change_by_step': pfb_relative_change_by_step,
        }
    finally:
        controller.active = False
        for block in model.unregistered_blocks:
            block.sa.kv_caching(False)


## CLIPSeg Mask Helpers


In [ ]:
from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor

CLIPSEG_PROCESSOR = None
CLIPSEG_MODEL = None
CONTENT_MASK_CACHE = {}
STYLE_MASK_CACHE = {}


def normalize_mask(mask):
    mask = mask.float()
    mask = mask - mask.min()
    return (mask / mask.max().clamp_min(1e-8)).clamp(0, 1)


def get_clipseg():
    global CLIPSEG_PROCESSOR, CLIPSEG_MODEL
    if CLIPSEG_PROCESSOR is None or CLIPSEG_MODEL is None:
        CLIPSEG_PROCESSOR = CLIPSegProcessor.from_pretrained(CLIPSEG_MODEL_ID)
        CLIPSEG_MODEL = CLIPSegForImageSegmentation.from_pretrained(CLIPSEG_MODEL_ID).to(CLIPSEG_DEVICE).eval()
    return CLIPSEG_PROCESSOR, CLIPSEG_MODEL


@torch.no_grad()
def segment_pil_with_clipseg(image, prompt, threshold=None):
    processor, model = get_clipseg()
    inputs = processor(text=[prompt], images=[image.convert('RGB')], padding=True, return_tensors='pt')
    inputs = {key: value.to(CLIPSEG_DEVICE) for key, value in inputs.items()}
    logits = model(**inputs).logits
    if logits.ndim == 2:
        logits = logits.unsqueeze(0)
    mask = torch.sigmoid(logits).unsqueeze(1)
    mask = F.interpolate(mask, size=(image.height, image.width), mode='bilinear', align_corners=False)
    mask = normalize_mask(mask[0, 0].detach().float().cpu())
    if threshold is not None:
        mask = (mask >= float(threshold)).float()
    return mask


def save_mask_image(mask, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(mask.clamp(0, 1).mul(255).byte().numpy(), mode='L').save(path)


def save_mask_overlay(image, mask, path, color=(255, 64, 64), alpha=0.45):
    pil = image if isinstance(image, Image.Image) else _image_tensor_to_pil(image)
    pil = pil.convert('RGB')
    mask_image = Image.fromarray(mask.clamp(0, 1).mul(255).byte().numpy(), mode='L').resize(pil.size)
    overlay = Image.new('RGB', pil.size, color)
    blended = Image.composite(Image.blend(pil, overlay, alpha), pil, mask_image)
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    blended.save(path)


def content_mask_path_for_pair(row):
    return MASK_DIR / f"{pair_slug(row)}_content_object_mask.png"


def content_mask_overlay_path_for_pair(row):
    return MASK_OVERLAY_DIR / f"{pair_slug(row)}_content_object_overlay.png"


def style_mask_path_for_pair(row):
    return STYLE_MASK_DIR / f"{pair_slug(row)}_style_object_mask.png"


def style_mask_overlay_path_for_pair(row):
    return STYLE_MASK_OVERLAY_DIR / f"{pair_slug(row)}_style_object_overlay.png"


def baseline_path_for_pair(row):
    return BASELINE_DIR / f"{pair_slug(row)}_baseline_content_stream.png"


## Shared Output Helpers


In [ ]:
def _safe_name(text):
    return ''.join(character if character.isalnum() else '_' for character in str(text)).strip('_').lower()


def _image_tensor_to_pil(image):
    tensor = image.detach().float().cpu()
    if tensor.ndim == 4:
        tensor = tensor[0]
    array = tensor.clamp(0, 1).permute(1, 2, 0).mul(255).byte().numpy()
    return Image.fromarray(array)


def _save_image_tensor(image, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    _image_tensor_to_pil(image).save(path)


def pair_slug(row):
    return f"{int(row['pair_id']):02d}_{row['content_id']}__STYLE__{row['style_id']}"


def improved_output_path_for_pair(row):
    return IMPROVED_DIR / f"{pair_slug(row)}_diagnostic_selected_object_masked.png"


def variant_output_path_for_pair(row, variant):
    if variant == 'diagnostic_selected_object_masked':
        return improved_output_path_for_pair(row)
    raise ValueError(f'Unknown variant: {variant}')


def trace_path_for_pair(row, variant):
    return TRACE_DIR / f"{pair_slug(row)}_{variant}_trace.pt"


def meta_path_for_pair(row, variant):
    return TRACE_DIR / f"{pair_slug(row)}_{variant}_meta.json"


def kingfisher_output_path_for_pair(row):
    return KINGFISHER_PAIR_OUTPUT_DIR / f"{pair_slug(row)}.png"


STYLE_IMAGE_CACHE = {}
STYLE_FEATURE_CACHE = {}


def get_style_image(style_path):
    key = str(Path(style_path))
    if key not in STYLE_IMAGE_CACHE:
        _, style_01, _ = load_reference_image(style_path, size=IMAGE_SIZE_HW[0])
        STYLE_IMAGE_CACHE[key] = style_01.detach().float().cpu()
    return STYLE_IMAGE_CACHE[key]


def get_style_features(style_path):
    key = str(Path(style_path))
    if key not in STYLE_FEATURE_CACHE:
        style_m11, _, _ = load_reference_image(style_path, size=IMAGE_SIZE_HW[0])
        with torch.inference_mode():
            features = extract_multiscale_style_features(style_m11)
        STYLE_FEATURE_CACHE[key] = [feature.detach().float().cpu() for feature in features]
        del style_m11, features
        gc.collect()
        torch.cuda.empty_cache()
    return [feature.to(device) for feature in STYLE_FEATURE_CACHE[key]]


def decode_feature_trace_to_pil_images(trace):
    images = []
    with torch.no_grad():
        for cumulative_codes in trace:
            image_01 = _decode_summed_codes_to_image_01(cumulative_codes.to(device))
            images.append(_image_tensor_to_pil(image_01))
    return images


def save_generation_artifacts(row, result, variant, output_path, config_payload):
    _save_image_tensor(result['stylized_image_01'].detach().float().cpu(), output_path)
    trace_payload = {
        'content_features': [feature.detach().float().cpu() for feature in result['content_features']],
        'style_features': [feature.detach().float().cpu() for feature in get_style_features(row['style_path'])],
        'generation_features': [feature.detach().float().cpu() for feature in result['generation_features']],
    }
    torch.save(trace_payload, trace_path_for_pair(row, variant))
    metadata = {
        'pair_id': int(row['pair_id']),
        'content_id': row['content_id'],
        'style_id': row['style_id'],
        'content_object': row['content_object'],
        'style_object': row['style_object'],
        'style_label': row['style_label'],
        'pfb_prompt': row['pfb_prompt'],
        'variant': variant,
        'variant_label': VARIANT_LABELS.get(variant, variant),
        'config': config_payload,
        'selected_object_steps': SELECTED_OBJECT_STEPS,
        'selected_feature_indices': SELECTED_FEATURE_INDICES,
        'seed': SEED + int(row['pair_id']),
        'cfg': CFG,
        'tau': TAU,
        'top_k': TOP_K,
        'top_p': TOP_P,
        'pfb_relative_change_by_step': result.get('pfb_relative_change_by_step'),
        'sac_calls': result.get('sac_calls'),
        'max_q_copy_error': result.get('max_q_copy_error'),
        'max_k_copy_error': result.get('max_k_copy_error'),
    }
    with open(meta_path_for_pair(row, variant), 'w') as f:
        json.dump(metadata, f, indent=2)
    return output_path


## Generate Baseline Content Images And Masks


In [ ]:
def generate_baseline_and_masks_for_pair(row, force=False):
    baseline_path = baseline_path_for_pair(row)
    content_mask_path = content_mask_path_for_pair(row)
    style_mask_path = style_mask_path_for_pair(row)

    if baseline_path.exists() and content_mask_path.exists() and style_mask_path.exists() and not force:
        content_mask = torchvision.transforms.functional.pil_to_tensor(Image.open(content_mask_path).convert('L')).float()[0] / 255.0
        style_mask = torchvision.transforms.functional.pil_to_tensor(Image.open(style_mask_path).convert('L')).float()[0] / 255.0
        return content_mask, style_mask

    with torch.inference_mode():
        baseline_result = paper_dual_path_generate(
            infinity,
            row['pfb_prompt'],
            [],
            seed=SEED + int(row['pair_id']),
            cfg=CFG,
            tau=TAU,
            top_k=TOP_K,
            top_p=TOP_P,
            pfb_feature_indices=[0],
            sac_prediction_start=SAC_START,
            edit_mode='none',
            enable_sac=False,
        )

    baseline_image = baseline_result['content_image_01'].detach().float().cpu()
    _save_image_tensor(baseline_image, baseline_path)

    content_pil = _image_tensor_to_pil(baseline_image)
    style_pil = Image.open(row['style_path']).convert('RGB')

    content_mask = segment_pil_with_clipseg(content_pil, row['content_object'], threshold=CONTENT_MASK_THRESHOLD)
    style_mask = segment_pil_with_clipseg(style_pil, STYLE_MASK_PROMPT, threshold=STYLE_MASK_THRESHOLD)

    save_mask_image(content_mask, content_mask_path)
    save_mask_overlay(content_pil, content_mask, content_mask_overlay_path_for_pair(row))
    save_mask_image(style_mask, style_mask_path)
    save_mask_overlay(style_pil, style_mask, style_mask_overlay_path_for_pair(row))

    del baseline_result
    gc.collect()
    torch.cuda.empty_cache()
    return content_mask, style_mask


def generate_all_baselines_and_masks(pair_rows=PAIR_ROWS, force=False):
    masks = {}
    for row in tqdm(pair_rows, desc='Baseline content images + CLIPSeg masks'):
        masks[int(row['pair_id'])] = generate_baseline_and_masks_for_pair(row, force=force)
    return masks

RUN_BASELINES_AND_MASKS = True
FORCE_REBUILD_MASKS = False

if RUN_BASELINES_AND_MASKS:
    MASKS_BY_PAIR_ID = generate_all_baselines_and_masks(PAIR_ROWS, force=FORCE_REBUILD_MASKS)
else:
    print('Set RUN_BASELINES_AND_MASKS = True to build baseline images and masks.')


## Region-Aware Scale Selection Diagnostic

Before generating the final samples, this diagnostic selects the top-k object-specific scales. For each candidate scale, it tests SVD-guided replacement on 50 CSD100 source/target pairs and scores whether the intervention improves **object-region style** while avoiding **background style spillover** and **style-object leakage**.


In [ ]:
# Diagnostic helpers for selecting object-specific injection scales.
from PIL import ImageFilter


def load_mask_tensor(mask_path):
    return torchvision.transforms.functional.pil_to_tensor(Image.open(mask_path).convert('L')).float()[0] / 255.0


def soften_mask_tensor(mask, dilation_size=15, blur_radius=8):
    mask = mask.detach().float().cpu().clamp(0, 1)
    mask_image = Image.fromarray((mask.numpy() * 255).round().astype('uint8'), mode='L')
    if dilation_size and dilation_size > 1:
        if dilation_size % 2 == 0:
            dilation_size += 1
        mask_image = mask_image.filter(ImageFilter.MaxFilter(int(dilation_size)))
    if blur_radius and blur_radius > 0:
        mask_image = mask_image.filter(ImageFilter.GaussianBlur(float(blur_radius)))
    return torchvision.transforms.functional.pil_to_tensor(mask_image).float()[0] / 255.0


def load_csd_image_for_diagnostic(path, size=512):
    image = Image.open(path).convert('RGB')
    image = ImageOps.fit(image, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    image_01 = torchvision.transforms.functional.to_tensor(image).unsqueeze(0).to(device)
    return image_01.mul(2).sub(1), image


@torch.no_grad()
def encode_residuals_for_diagnostic(image_m11):
    with torch.amp.autocast('cuda', enabled=False):
        _, _, _, all_bit_indices, _, _ = vae.encode(image_m11.float(), scale_schedule=SCALE_SCHEDULE)
    residuals = []
    final_size = SCALE_SCHEDULE[-1]
    for step_id, bit_indices in enumerate(all_bit_indices):
        codes = vae.quantizer.lfq.indices_to_codes(bit_indices, label_type='bit_label')
        if step_id != len(SCALE_SCHEDULE) - 1:
            codes = F.interpolate(codes, size=final_size, mode=vae.quantizer.z_interplote_up)
        residuals.append(codes.detach().float().cpu())
    return residuals


def sum_residuals_for_diagnostic(residuals, start=0, stop=None):
    stop = len(residuals) if stop is None else stop
    selected = residuals[start:stop]
    if not selected:
        return torch.zeros_like(residuals[0])
    return torch.stack([residual.float() for residual in selected], dim=0).sum(dim=0)


def phi_svd_for_diagnostic(feature, alpha=1.0, rank=1):
    original_dtype = feature.dtype
    batch, channels = feature.shape[:2]
    spatial_shape = feature.shape[2:]
    outputs = []
    for batch_id in range(batch):
        matrix = feature[batch_id].detach().float().reshape(channels, -1)
        u, singular_values, vh = torch.linalg.svd(matrix, full_matrices=False)
        used_rank = singular_values.numel() if rank is None else min(int(rank), singular_values.numel())
        weights = torch.exp(-float(PAPER_ALPHA) * torch.arange(used_rank, device=matrix.device, dtype=matrix.dtype))
        weighted_s = singular_values[:used_rank] * weights
        reconstructed = (u[:, :used_rank] * weighted_s.unsqueeze(0)) @ vh[:used_rank]
        outputs.append(reconstructed.reshape(channels, *spatial_shape))
    return torch.stack(outputs).to(dtype=original_dtype)


@torch.no_grad()
def reconstruct_diagnostic_svd(source_residuals, target_residuals, scale_index, rank=1):
    source_prefix = sum_residuals_for_diagnostic(source_residuals, stop=scale_index + 1)
    target_prefix = sum_residuals_for_diagnostic(target_residuals, stop=scale_index + 1)
    target_tail = sum_residuals_for_diagnostic(target_residuals, start=scale_index + 1)
    edited_prefix = phi_svd_for_diagnostic(source_prefix, rank=rank) + (
        target_prefix - phi_svd_for_diagnostic(target_prefix, rank=rank)
    )
    image_01 = _decode_summed_codes_to_image_01((edited_prefix + target_tail).to(device))
    return _image_tensor_to_pil(image_01)


@torch.no_grad()
def reconstruct_diagnostic_baseline(target_residuals):
    image_01 = _decode_summed_codes_to_image_01(sum_residuals_for_diagnostic(target_residuals).to(device))
    return _image_tensor_to_pil(image_01)


def masked_clip_image(image, mask, region='object', crop=False):
    image = image.convert('RGB')
    mask_image = Image.fromarray((mask.detach().float().cpu().clamp(0, 1).numpy() * 255).round().astype('uint8'), mode='L')
    mask_image = mask_image.resize(image.size, Image.Resampling.BILINEAR)
    if region == 'background':
        mask_image = Image.eval(mask_image, lambda value: 255 - value)
    white = Image.new('RGB', image.size, 'white')
    composed = Image.composite(image, white, mask_image)
    if crop and region == 'object':
        bbox = mask_image.point(lambda value: 255 if value > 32 else 0).getbbox()
        if bbox is not None:
            composed = composed.crop(bbox)
    return composed


In [ ]:
# CLIP helpers for the scale-selection diagnostic.
from transformers import CLIPModel, CLIPProcessor

DIAG_CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'
DIAG_CLIP_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DIAG_CLIP_PROCESSOR = None
DIAG_CLIP_MODEL = None
DIAG_TEXT_CACHE = {}
DIAG_IMAGE_CACHE = {}


def get_diag_clip_model():
    global DIAG_CLIP_PROCESSOR, DIAG_CLIP_MODEL
    if DIAG_CLIP_PROCESSOR is None or DIAG_CLIP_MODEL is None:
        DIAG_CLIP_PROCESSOR = CLIPProcessor.from_pretrained(DIAG_CLIP_MODEL_ID)
        DIAG_CLIP_MODEL = CLIPModel.from_pretrained(DIAG_CLIP_MODEL_ID).to(DIAG_CLIP_DEVICE).eval()
    return DIAG_CLIP_PROCESSOR, DIAG_CLIP_MODEL


def diag_clip_feature_tensor(output, projection_layer=None):
    if torch.is_tensor(output):
        return output
    if hasattr(output, 'image_embeds') and output.image_embeds is not None:
        return output.image_embeds
    if hasattr(output, 'text_embeds') and output.text_embeds is not None:
        return output.text_embeds
    if hasattr(output, 'pooler_output') and output.pooler_output is not None:
        pooled = output.pooler_output
        if projection_layer is not None and pooled.shape[-1] == projection_layer.in_features:
            return projection_layer(pooled)
        return pooled
    if isinstance(output, (tuple, list)) and output:
        first = output[0]
        if torch.is_tensor(first):
            return first
    raise TypeError(f'Could not convert diagnostic CLIP output to a feature tensor: {type(output)}')


@torch.no_grad()
def diag_clip_text_embedding(text):
    key = str(text)
    if key in DIAG_TEXT_CACHE:
        return DIAG_TEXT_CACHE[key]
    processor, model = get_diag_clip_model()
    inputs = processor(text=[key], return_tensors='pt', padding=True, truncation=True)
    inputs = {name: tensor.to(DIAG_CLIP_DEVICE) for name, tensor in inputs.items()}
    output = model.get_text_features(**inputs)
    embedding = diag_clip_feature_tensor(output, getattr(model, 'text_projection', None)).float()
    embedding = F.normalize(embedding, dim=-1).detach().cpu()
    DIAG_TEXT_CACHE[key] = embedding
    return embedding


@torch.no_grad()
def diag_clip_image_embedding(image, cache_key=None):
    if cache_key is not None and cache_key in DIAG_IMAGE_CACHE:
        return DIAG_IMAGE_CACHE[cache_key]
    processor, model = get_diag_clip_model()
    inputs = processor(images=image.convert('RGB'), return_tensors='pt')
    inputs = {name: tensor.to(DIAG_CLIP_DEVICE) for name, tensor in inputs.items()}
    output = model.get_image_features(**inputs)
    embedding = diag_clip_feature_tensor(output, getattr(model, 'visual_projection', None)).float()
    embedding = F.normalize(embedding, dim=-1).detach().cpu()
    if cache_key is not None:
        DIAG_IMAGE_CACHE[cache_key] = embedding
    return embedding


def diag_cosine(image_embedding, text_embedding):
    return float(F.cosine_similarity(image_embedding.float(), text_embedding.float(), dim=-1).item())


In [ ]:
# Run the top-k scale selection diagnostic.
def parse_csd100_item_for_diagnostic(folder):
    folder = Path(folder)
    object_label, style_label = folder.name.split('+', 1)
    return {
        'item_id': folder.name,
        'image_path': folder / '00.jpg',
        'object_label': object_label.replace('_', ' ').replace('-', ' '),
        'style_label': style_label.replace('_', ' ').replace('-', ' '),
    }


def build_diagnostic_pairs(num_pairs=SCALE_SELECTION_NUM_PAIRS):
    items = [parse_csd100_item_for_diagnostic(path) for path in sorted(CSD100_DIR.iterdir()) if path.is_dir() and (path / '00.jpg').exists()]
    rng = random.Random(SEED + 1800)
    rng.shuffle(items)
    rows = []
    cursor = 0
    while len(rows) < num_pairs and cursor + 1 < len(items):
        source = items[cursor]
        target = items[cursor + 1]
        cursor += 2
        if source['item_id'] == target['item_id']:
            continue
        rows.append({
            'diag_pair_id': len(rows),
            'source_id': source['item_id'],
            'source_path': source['image_path'],
            'source_object': source['object_label'],
            'source_style': source['style_label'],
            'target_id': target['item_id'],
            'target_path': target['image_path'],
            'target_object': target['object_label'],
            'target_style': target['style_label'],
            'source_style_text': f"a {source['style_label']} style image",
            'source_object_text': f"a photo of {source['object_label']}",
        })
    if len(rows) < num_pairs:
        raise RuntimeError(f'Only built {len(rows)} diagnostic pairs; requested {num_pairs}.')
    return rows


def compute_scale_selection_scores(force=False):
    metrics_path = SCALE_SELECTION_DIR / 'scale_selection_region_metrics.csv'
    summary_path = SCALE_SELECTION_DIR / 'scale_selection_summary.csv'
    selected_path = SCALE_SELECTION_DIR / 'selected_object_steps.json'
    if metrics_path.exists() and summary_path.exists() and selected_path.exists() and not force:
        metrics_df = pd.read_csv(metrics_path)
        summary_df = pd.read_csv(summary_path)
        selected_payload = json.loads(selected_path.read_text())
        return metrics_df, summary_df, selected_payload

    diagnostic_rows = build_diagnostic_pairs(SCALE_SELECTION_NUM_PAIRS)
    metric_rows = []
    for row in tqdm(diagnostic_rows, desc='Selecting object injection scales'):
        source_m11, source_pil = load_csd_image_for_diagnostic(row['source_path'], size=IMAGE_SIZE_HW[0])
        target_m11, target_pil = load_csd_image_for_diagnostic(row['target_path'], size=IMAGE_SIZE_HW[0])
        source_residuals = encode_residuals_for_diagnostic(source_m11)
        target_residuals = encode_residuals_for_diagnostic(target_m11)
        baseline_image = reconstruct_diagnostic_baseline(target_residuals)

        target_mask = segment_pil_with_clipseg(target_pil, row['target_object'], threshold=CONTENT_MASK_THRESHOLD)
        soft_target_mask = soften_mask_tensor(target_mask, dilation_size=SOFT_MASK_DILATION_SIZE, blur_radius=SOFT_MASK_BLUR_RADIUS)

        style_text_embedding = diag_clip_text_embedding(row['source_style_text'])
        source_object_embedding = diag_clip_text_embedding(row['source_object_text'])

        baseline_object = masked_clip_image(baseline_image, soft_target_mask, region='object', crop=True)
        baseline_background = masked_clip_image(baseline_image, soft_target_mask, region='background', crop=False)
        baseline_object_style = diag_cosine(diag_clip_image_embedding(baseline_object), style_text_embedding)
        baseline_background_style = diag_cosine(diag_clip_image_embedding(baseline_background), style_text_embedding)
        baseline_leakage = diag_cosine(diag_clip_image_embedding(baseline_object), source_object_embedding)

        for scale_index in CANDIDATE_OBJECT_STEPS:
            svd_image = reconstruct_diagnostic_svd(source_residuals, target_residuals, scale_index, rank=OBJECT_FOREGROUND_RANK)
            svd_object = masked_clip_image(svd_image, soft_target_mask, region='object', crop=True)
            svd_background = masked_clip_image(svd_image, soft_target_mask, region='background', crop=False)

            object_style = diag_cosine(diag_clip_image_embedding(svd_object), style_text_embedding)
            background_style = diag_cosine(diag_clip_image_embedding(svd_background), style_text_embedding)
            leakage = diag_cosine(diag_clip_image_embedding(svd_object), source_object_embedding)

            object_style_gain = object_style - baseline_object_style
            background_style_gain = background_style - baseline_background_style
            leakage_gain = leakage - baseline_leakage
            score = (
                object_style_gain
                - OBJECT_SCORE_BACKGROUND_WEIGHT * max(0.0, background_style_gain)
                - OBJECT_SCORE_LEAKAGE_WEIGHT * max(0.0, leakage_gain)
            )
            metric_rows.append({
                'diag_pair_id': int(row['diag_pair_id']),
                'source_id': row['source_id'],
                'target_id': row['target_id'],
                'source_object': row['source_object'],
                'source_style': row['source_style'],
                'target_object': row['target_object'],
                'scale_index_zero_based': int(scale_index),
                'object_style_gain': object_style_gain,
                'background_style_gain': background_style_gain,
                'style_object_leakage_gain': leakage_gain,
                'object_injection_score': score,
            })

        del source_m11, target_m11, source_residuals, target_residuals
        gc.collect()
        torch.cuda.empty_cache()

    metrics_df = pd.DataFrame(metric_rows)
    summary_df = metrics_df.groupby('scale_index_zero_based', as_index=False).agg(
        object_style_gain=('object_style_gain', 'mean'),
        background_style_gain=('background_style_gain', 'mean'),
        style_object_leakage_gain=('style_object_leakage_gain', 'mean'),
        object_injection_score=('object_injection_score', 'mean'),
    ).sort_values('object_injection_score', ascending=False)

    selected_steps = summary_df.head(SELECT_TOP_K_OBJECT_STEPS)['scale_index_zero_based'].astype(int).tolist()
    selected_steps = sorted(selected_steps)
    selected_payload = {
        'selected_object_steps': selected_steps,
        'candidate_object_steps': CANDIDATE_OBJECT_STEPS,
        'top_k': SELECT_TOP_K_OBJECT_STEPS,
        'background_weight': OBJECT_SCORE_BACKGROUND_WEIGHT,
        'leakage_weight': OBJECT_SCORE_LEAKAGE_WEIGHT,
        'foreground_rank_used_for_selection': OBJECT_FOREGROUND_RANK,
    }
    metrics_df.to_csv(metrics_path, index=False)
    summary_df.to_csv(summary_path, index=False)
    selected_path.write_text(json.dumps(selected_payload, indent=2))
    print('Saved diagnostic metrics:', metrics_path)
    print('Saved diagnostic summary:', summary_path)
    print('Saved selected steps:', selected_path)
    return metrics_df, summary_df, selected_payload


RUN_SCALE_SELECTION_DIAGNOSTIC = True
FORCE_RERUN_SCALE_SELECTION = False

if RUN_SCALE_SELECTION_DIAGNOSTIC:
    SCALE_SELECTION_METRICS_DF, SCALE_SELECTION_SUMMARY_DF, SELECTED_SCALE_PAYLOAD = compute_scale_selection_scores(
        force=FORCE_RERUN_SCALE_SELECTION,
    )
    SELECTED_OBJECT_STEPS = [int(step) for step in SELECTED_SCALE_PAYLOAD['selected_object_steps']]
    SELECTED_FEATURE_INDICES = sorted(set(GLOBAL_STYLE_STEPS + SELECTED_OBJECT_STEPS))
    SELECTED_STYLE_STRENGTH_BY_STEP = dict(GLOBAL_STYLE_STRENGTH_BY_STEP)
    for order, step in enumerate(SELECTED_OBJECT_STEPS):
        SELECTED_STYLE_STRENGTH_BY_STEP[int(step)] = float(OBJECT_FOREGROUND_STRENGTH * (OBJECT_FOREGROUND_DECAY ** order))
    VARIANT_INJECTED_INDICES['diagnostic_selected_object_masked'] = SELECTED_FEATURE_INDICES

    display(SCALE_SELECTION_SUMMARY_DF)
    print('Selected object steps:', SELECTED_OBJECT_STEPS)
    print('All injection steps:', SELECTED_FEATURE_INDICES)
    print('Strength by step:', SELECTED_STYLE_STRENGTH_BY_STEP)

    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    plot_df = SCALE_SELECTION_SUMMARY_DF.sort_values('scale_index_zero_based')
    ax.plot(plot_df['scale_index_zero_based'], plot_df['object_style_gain'], marker='o', label='object style gain')
    ax.plot(plot_df['scale_index_zero_based'], plot_df['background_style_gain'], marker='s', label='background style gain')
    ax.plot(plot_df['scale_index_zero_based'], plot_df['style_object_leakage_gain'], marker='^', label='style-object leakage gain')
    ax.plot(plot_df['scale_index_zero_based'], plot_df['object_injection_score'], marker='D', linewidth=2.8, label='selection score')
    for step in SELECTED_OBJECT_STEPS:
        ax.axvline(step, color='black', linestyle='--', alpha=0.2)
    ax.set_xlabel('candidate object scale index')
    ax.set_ylabel('mean CLIP delta / score')
    ax.set_title('Region-aware diagnostic for object-style injection scale selection')
    ax.grid(True, alpha=0.25)
    ax.legend(loc='best')
    fig.tight_layout()
    selection_plot_path = SCALE_SELECTION_DIR / 'scale_selection_summary.png'
    fig.savefig(selection_plot_path, dpi=180, bbox_inches='tight')
    print('Saved selection plot:', selection_plot_path)
    plt.show()
else:
    print('Set RUN_SCALE_SELECTION_DIAGNOSTIC = True to select object-injection scales.')


## Generate Improved Variant

The selected object scales are now used in the final generation variant on the fixed 20 CSD100 pairs.


In [ ]:
def generate_one_pair_diagnostic_selected(row, force=False):
    variant = 'diagnostic_selected_object_masked'
    if not SELECTED_OBJECT_STEPS:
        raise RuntimeError('SELECTED_OBJECT_STEPS is empty. Run the scale-selection diagnostic first.')

    output_path = improved_output_path_for_pair(row)
    if output_path.exists() and trace_path_for_pair(row, variant).exists() and not force:
        print('Already exists:', output_path)
        return output_path

    if not content_mask_path_for_pair(row).exists() or not style_mask_path_for_pair(row).exists():
        content_mask, style_mask = generate_baseline_and_masks_for_pair(row, force=False)
    else:
        content_mask = load_mask_tensor(content_mask_path_for_pair(row))
        style_mask = load_mask_tensor(style_mask_path_for_pair(row))

    content_mask = soften_mask_tensor(content_mask, dilation_size=SOFT_MASK_DILATION_SIZE, blur_radius=SOFT_MASK_BLUR_RADIUS)
    style_mask = soften_mask_tensor(style_mask, dilation_size=SOFT_MASK_DILATION_SIZE, blur_radius=SOFT_MASK_BLUR_RADIUS)

    feature_masks_by_step = {step: content_mask for step in SELECTED_OBJECT_STEPS}
    style_masks_by_step = {step: style_mask for step in SELECTED_OBJECT_STEPS}

    style_features = get_style_features(row['style_path'])
    with torch.inference_mode():
        result = paper_dual_path_generate(
            infinity,
            row['pfb_prompt'],
            style_features,
            seed=SEED + int(row['pair_id']),
            cfg=CFG,
            tau=TAU,
            top_k=TOP_K,
            top_p=TOP_P,
            pfb_feature_indices=SELECTED_FEATURE_INDICES,
            sac_prediction_start=SAC_START,
            edit_mode='pfb',
            alpha=PAPER_ALPHA,
            rank=GLOBAL_SVD_RANK,
            style_strength=1.0,
            style_decay=1.0,
            style_strength_by_step=SELECTED_STYLE_STRENGTH_BY_STEP,
            feature_masks_by_step=feature_masks_by_step,
            style_masks_by_step=style_masks_by_step,
            masked_background_strength=MASKED_BACKGROUND_STRENGTH,
            split_style_regions=True,
            foreground_rank=OBJECT_FOREGROUND_RANK,
            background_rank=OBJECT_BACKGROUND_RANK,
            sac_strength=1.0,
            enable_sac=True,
        )

    save_generation_artifacts(
        row,
        result,
        variant,
        output_path,
        {
            'formula': 'global PFB at 0,1,2 plus diagnostic-selected soft object-mask PFB at top-k object scales',
            'global_steps': GLOBAL_STYLE_STEPS,
            'selected_object_steps': SELECTED_OBJECT_STEPS,
            'feature_indices': SELECTED_FEATURE_INDICES,
            'global_rank': GLOBAL_SVD_RANK,
            'object_foreground_rank': OBJECT_FOREGROUND_RANK,
            'object_background_rank': OBJECT_BACKGROUND_RANK,
            'strength_by_step': SELECTED_STYLE_STRENGTH_BY_STEP,
            'masked_background_strength': MASKED_BACKGROUND_STRENGTH,
            'soft_mask_dilation_size': SOFT_MASK_DILATION_SIZE,
            'soft_mask_blur_radius': SOFT_MASK_BLUR_RADIUS,
            'scale_selection_score': 'object_style_gain - bg_weight * positive_background_gain - leakage_weight * positive_leakage_gain',
            'scale_selection_summary': str(SCALE_SELECTION_DIR / 'scale_selection_summary.csv'),
            'content_mask_prompt': row['content_object'],
            'style_mask_prompt': STYLE_MASK_PROMPT,
        },
    )
    print('Saved:', output_path)
    del style_features, result
    gc.collect()
    torch.cuda.empty_cache()
    return output_path


def run_all_diagnostic_selected(pair_rows=PAIR_ROWS, force=False):
    outputs = []
    for row in tqdm(pair_rows, desc='Improved variant: diagnostic-selected object-mask PFB + SAC'):
        outputs.append(generate_one_pair_diagnostic_selected(row, force=force))
    return outputs

RUN_DIAGNOSTIC_SELECTED_VARIANT = True
FORCE_REGENERATE_DIAGNOSTIC_SELECTED_VARIANT = True

if RUN_DIAGNOSTIC_SELECTED_VARIANT:
    DIAGNOSTIC_SELECTED_OUTPUTS = run_all_diagnostic_selected(
        PAIR_ROWS,
        force=FORCE_REGENERATE_DIAGNOSTIC_SELECTED_VARIANT,
    )
else:
    print('Set RUN_DIAGNOSTIC_SELECTED_VARIANT = True to generate the improved variant.')


## Scale Trajectory Diagnostics


In [ ]:
def load_trace(row, variant):
    path = trace_path_for_pair(row, variant)
    if not path.exists():
        raise FileNotFoundError(path)
    return torch.load(path, map_location='cpu')


def plot_scale_diagnostic(row, variant, save=True):
    title = VARIANT_LABELS[variant]
    injected_indices = VARIANT_INJECTED_INDICES[variant]
    trace_payload = load_trace(row, variant)
    traces = [
        trace_payload['content_features'],
        trace_payload['style_features'],
        trace_payload['generation_features'],
    ]
    row_labels = [
        f"Content stream\n{row['pfb_prompt']}",
        f"Style reference\n{row['style_object']}\n({row['style_label']})",
        title,
    ]
    decoded_rows = [decode_feature_trace_to_pil_images(trace) for trace in traces]
    num_scales = min(len(images) for images in decoded_rows)

    fig, axes = plt.subplots(3, num_scales + 1, figsize=(1.55 * (num_scales + 1), 5.2), squeeze=False)
    for axis in axes.reshape(-1):
        axis.axis('off')
    for row_id, (label, images) in enumerate(zip(row_labels, decoded_rows)):
        axes[row_id, 0].text(0.5, 0.5, label, ha='center', va='center', fontsize=9, wrap=True)
        for scale_id in range(num_scales):
            axes[row_id, scale_id + 1].imshow(images[scale_id])
            if row_id == 0:
                marker = '*' if scale_id in injected_indices else ''
                axes[row_id, scale_id + 1].set_title(f'R{scale_id + 1}{marker}', fontsize=8)
    fig.suptitle(f"Pair {int(row['pair_id']):02d}: {row['content_id']} -> {row['style_id']} | {title}", fontsize=11)
    fig.tight_layout()
    if save:
        save_path = DIAGNOSTIC_DIR / f"{pair_slug(row)}_{variant}_scale_diagnostic.png"
        fig.savefig(save_path, dpi=180, bbox_inches='tight')
        print('Saved diagnostic:', save_path)
    plt.show()
    plt.close(fig)


def plot_all_scale_diagnostics(pair_rows=PAIR_ROWS, variants=VARIANT_ORDER):
    for row in tqdm(pair_rows, desc='Scale diagnostics'):
        for variant in variants:
            if trace_path_for_pair(row, variant).exists():
                plot_scale_diagnostic(row, variant, save=True)
            else:
                print('Missing trace, skipping diagnostic:', pair_slug(row), variant)
        gc.collect()
        torch.cuda.empty_cache()

RUN_SCALE_DIAGNOSTICS = True

if RUN_SCALE_DIAGNOSTICS:
    plot_all_scale_diagnostics(PAIR_ROWS)
else:
    print('Set RUN_SCALE_DIAGNOSTICS = True after generation to plot scale diagnostics.')


## Final Comparison Gallery


In [ ]:
def _show_pil_or_placeholder(axis, image_path, missing_text):
    if Path(image_path).exists():
        axis.imshow(Image.open(image_path).convert('RGB'))
    else:
        axis.text(0.5, 0.5, missing_text, ha='center', va='center', fontsize=9, wrap=True)
    axis.axis('off')


def show_notebook18_gallery(pair_rows=PAIR_ROWS, variants=VARIANT_ORDER):
    rows = [row for row in pair_rows if any(variant_output_path_for_pair(row, variant).exists() for variant in variants)]
    if not rows:
        print('No generated images found yet.')
        return

    columns = ['content source', 'style reference', 'baseline content stream'] + [VARIANT_LABELS[variant] for variant in variants]
    fig, axes = plt.subplots(len(rows), len(columns), figsize=(3.2 * len(columns), 3.35 * len(rows)), squeeze=False)
    for row_id, row in enumerate(rows):
        image_paths = [row['content_path'], row['style_path'], baseline_path_for_pair(row)] + [
            variant_output_path_for_pair(row, variant) for variant in variants
        ]
        titles = [
            f"content: {row['content_object']}\nsource style: {row['content_style']}",
            f"style: {row['style_label']}\nobject: {row['style_object']}",
            row['pfb_prompt'],
        ] + [VARIANT_LABELS[variant] for variant in variants]
        missing = ['missing content', 'missing style', 'missing baseline'] + [f'missing {variant}' for variant in variants]
        for col_id, (path, title, missing_text) in enumerate(zip(image_paths, titles, missing)):
            _show_pil_or_placeholder(axes[row_id, col_id], path, missing_text)
            if row_id == 0:
                axes[row_id, col_id].set_title(f"{columns[col_id]}\n{title}", fontsize=8)
            else:
                axes[row_id, col_id].set_title(title, fontsize=8)
    fig.suptitle('Notebook 18: diagnostic-selected object-masked PFB + SAC', fontsize=14)
    fig.tight_layout()
    gallery_path = GALLERY_DIR / 'notebook18_diagnostic_selected_gallery.png'
    fig.savefig(gallery_path, dpi=180, bbox_inches='tight')
    print('Saved gallery:', gallery_path)
    plt.show()

show_notebook18_gallery(PAIR_ROWS)


## Evaluation Metrics: `S_txt`, `S_img`, `S_harmonic`

This section scores each generated image with CLIP:

```text
S_txt      = CLIP(generated image, style text prompt)
S_img      = CLIP image-image similarity between generated image and style reference image
S_harmonic = 2 * S_txt * S_img / (S_txt + S_img)
```

The harmonic score is useful because it punishes imbalance: a result needs to match both the named style and the style reference image.


In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_EVAL_MODEL_ID = 'openai/clip-vit-base-patch32'
CLIP_EVAL_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIP_EVAL_PROCESSOR = None
CLIP_EVAL_MODEL = None
CLIP_IMAGE_EMBED_CACHE = {}
CLIP_TEXT_EMBED_CACHE = {}


def get_clip_eval_model():
    global CLIP_EVAL_PROCESSOR, CLIP_EVAL_MODEL
    if CLIP_EVAL_PROCESSOR is None or CLIP_EVAL_MODEL is None:
        CLIP_EVAL_PROCESSOR = CLIPProcessor.from_pretrained(CLIP_EVAL_MODEL_ID)
        CLIP_EVAL_MODEL = CLIPModel.from_pretrained(CLIP_EVAL_MODEL_ID).to(CLIP_EVAL_DEVICE).eval()
    return CLIP_EVAL_PROCESSOR, CLIP_EVAL_MODEL


def _clip_feature_tensor(output, projection_layer=None):
    if torch.is_tensor(output):
        return output
    if hasattr(output, 'image_embeds') and output.image_embeds is not None:
        return output.image_embeds
    if hasattr(output, 'text_embeds') and output.text_embeds is not None:
        return output.text_embeds
    if hasattr(output, 'pooler_output') and output.pooler_output is not None:
        pooled = output.pooler_output
        if projection_layer is not None and pooled.shape[-1] == projection_layer.in_features:
            return projection_layer(pooled)
        return pooled
    if isinstance(output, (tuple, list)) and output:
        first = output[0]
        if torch.is_tensor(first):
            return first
    raise TypeError(f'Could not convert CLIP output to a feature tensor: {type(output)}')


@torch.no_grad()
def clip_image_embedding(image_path):
    image_path = Path(image_path)
    key = str(image_path.resolve())
    if key in CLIP_IMAGE_EMBED_CACHE:
        return CLIP_IMAGE_EMBED_CACHE[key]
    processor, model = get_clip_eval_model()
    image = Image.open(image_path).convert('RGB')
    inputs = processor(images=image, return_tensors='pt')
    inputs = {name: tensor.to(CLIP_EVAL_DEVICE) for name, tensor in inputs.items()}

    output = model.get_image_features(**inputs)
    embedding = _clip_feature_tensor(output, getattr(model, 'visual_projection', None)).float()
    embedding = F.normalize(embedding, dim=-1)
    CLIP_IMAGE_EMBED_CACHE[key] = embedding.detach().cpu()
    return CLIP_IMAGE_EMBED_CACHE[key]


@torch.no_grad()
def clip_text_embedding(text):
    key = str(text)
    if key in CLIP_TEXT_EMBED_CACHE:
        return CLIP_TEXT_EMBED_CACHE[key]
    processor, model = get_clip_eval_model()
    inputs = processor(text=[text], return_tensors='pt', padding=True, truncation=True)
    inputs = {name: tensor.to(CLIP_EVAL_DEVICE) for name, tensor in inputs.items()}

    output = model.get_text_features(**inputs)
    embedding = _clip_feature_tensor(output, getattr(model, 'text_projection', None)).float()
    embedding = F.normalize(embedding, dim=-1)
    CLIP_TEXT_EMBED_CACHE[key] = embedding.detach().cpu()
    return CLIP_TEXT_EMBED_CACHE[key]


def cosine_score(embedding_a, embedding_b):
    score = F.cosine_similarity(embedding_a.float(), embedding_b.float(), dim=-1).item()
    # Keep the harmonic mean numerically meaningful; CLIP scores are usually positive here.
    return max(0.0, float(score))


def style_text_prompt(row):
    return f"a {row['style_label']} style image"


def compute_style_metrics_for_output(row, variant):
    output_path = variant_output_path_for_pair(row, variant)
    if not output_path.exists():
        return None

    generated_embedding = clip_image_embedding(output_path)
    style_image_embedding = clip_image_embedding(row['style_path'])
    style_text_embedding = clip_text_embedding(style_text_prompt(row))

    s_txt = cosine_score(generated_embedding, style_text_embedding)
    s_img = cosine_score(generated_embedding, style_image_embedding)
    s_harmonic = (2 * s_txt * s_img / (s_txt + s_img)) if (s_txt + s_img) > 0 else 0.0

    return {
        'pair_id': int(row['pair_id']),
        'content_id': row['content_id'],
        'style_id': row['style_id'],
        'content_object': row['content_object'],
        'style_object': row['style_object'],
        'style_label': row['style_label'],
        'variant': variant,
        'variant_label': VARIANT_LABELS[variant].replace('\n', ' '),
        'style_text_prompt': style_text_prompt(row),
        'S_txt': s_txt,
        'S_img': s_img,
        'S_harmonic': s_harmonic,
        'output_path': str(output_path),
    }


def compute_style_metrics(pair_rows=PAIR_ROWS, variants=VARIANT_ORDER):
    rows = []
    for row in tqdm(pair_rows, desc='CLIP style metrics'):
        for variant in variants:
            metrics = compute_style_metrics_for_output(row, variant)
            if metrics is not None:
                rows.append(metrics)
    metrics_df = pd.DataFrame(rows)
    if metrics_df.empty:
        print('No generated outputs found for metric evaluation.')
        return metrics_df

    metrics_path = EVAL_DIR / 'notebook18_style_metrics.csv'
    metrics_df.to_csv(metrics_path, index=False)
    print('Saved metrics:', metrics_path)

    summary = metrics_df.groupby(['variant', 'variant_label'], as_index=False)[['S_txt', 'S_img', 'S_harmonic']].mean()
    summary_path = EVAL_DIR / 'notebook18_style_metrics_summary.csv'
    summary.to_csv(summary_path, index=False)
    print('Saved summary:', summary_path)

    display(metrics_df)
    display(summary)

    fig, axis = plt.subplots(figsize=(9, 4.6))
    x = np.arange(len(summary))
    width = 0.24
    for offset, metric_name in [(-width, 'S_txt'), (0, 'S_img'), (width, 'S_harmonic')]:
        axis.bar(x + offset, summary[metric_name], width=width, label=metric_name)
    axis.set_xticks(x)
    axis.set_xticklabels(summary['variant_label'], rotation=12, ha='right')
    axis.set_ylim(0, max(0.35, float(summary[['S_txt', 'S_img', 'S_harmonic']].max().max()) * 1.15))
    axis.set_ylabel('CLIP cosine similarity')
    axis.set_title('Notebook 18 style metrics for diagnostic-selected variant')
    axis.grid(axis='y', alpha=0.25)
    axis.legend()
    fig.tight_layout()
    plot_path = EVAL_DIR / 'notebook18_style_metrics_summary.png'
    fig.savefig(plot_path, dpi=180, bbox_inches='tight')
    print('Saved metric plot:', plot_path)
    plt.show()

    return metrics_df


RUN_STYLE_METRICS = True

if RUN_STYLE_METRICS:
    STYLE_METRICS_DF = compute_style_metrics(PAIR_ROWS)
else:
    print('Set RUN_STYLE_METRICS = True after generation to compute S_txt, S_img, and S_harmonic.')


## Content Preservation and Style-Object Leakage Metrics

This separate diagnostic checks whether the generated image still looks like the target content object and whether it accidentally copied the style reference object:

```text
C_txt         = CLIP(generated image, content object prompt)
Leak          = CLIP(generated image, style object prompt)
ContentMargin = C_txt - Leak
```

Higher `C_txt` and `ContentMargin` are better. Lower `Leak` is better.


In [ ]:
def content_text_prompt(row):
    return row['pfb_prompt']


def style_object_text_prompt(row):
    return f"a photo of {row['style_object']}"


def compute_content_leakage_metrics_for_output(row, variant):
    output_path = variant_output_path_for_pair(row, variant)
    if not output_path.exists():
        return None

    generated_embedding = clip_image_embedding(output_path)
    content_text_embedding = clip_text_embedding(content_text_prompt(row))
    style_object_embedding = clip_text_embedding(style_object_text_prompt(row))

    c_txt = cosine_score(generated_embedding, content_text_embedding)
    leak = cosine_score(generated_embedding, style_object_embedding)
    content_margin = c_txt - leak

    return {
        'pair_id': int(row['pair_id']),
        'content_id': row['content_id'],
        'style_id': row['style_id'],
        'content_object': row['content_object'],
        'style_object': row['style_object'],
        'style_label': row['style_label'],
        'variant': variant,
        'variant_label': VARIANT_LABELS[variant].replace('\n', ' '),
        'content_text_prompt': content_text_prompt(row),
        'style_object_text_prompt': style_object_text_prompt(row),
        'C_txt': c_txt,
        'Leak': leak,
        'ContentMargin': content_margin,
        'output_path': str(output_path),
    }


def compute_content_leakage_metrics(pair_rows=PAIR_ROWS, variants=VARIANT_ORDER):
    rows = []
    for row in tqdm(pair_rows, desc='CLIP content/leakage metrics'):
        for variant in variants:
            metrics = compute_content_leakage_metrics_for_output(row, variant)
            if metrics is not None:
                rows.append(metrics)

    metrics_df = pd.DataFrame(rows)
    if metrics_df.empty:
        print('No generated outputs found for content/leakage metric evaluation.')
        return metrics_df

    metrics_path = EVAL_DIR / 'notebook18_content_leakage_metrics.csv'
    metrics_df.to_csv(metrics_path, index=False)
    print('Saved content/leakage metrics:', metrics_path)

    summary = metrics_df.groupby(
        ['variant', 'variant_label'],
        as_index=False,
    )[['C_txt', 'Leak', 'ContentMargin']].mean()

    summary_path = EVAL_DIR / 'notebook18_content_leakage_metrics_summary.csv'
    summary.to_csv(summary_path, index=False)
    print('Saved content/leakage summary:', summary_path)

    display(metrics_df)
    display(summary)

    fig, axis = plt.subplots(figsize=(9, 4.6))
    x = np.arange(len(summary))
    width = 0.24

    for offset, metric_name in [(-width, 'C_txt'), (0, 'Leak'), (width, 'ContentMargin')]:
        axis.bar(x + offset, summary[metric_name], width=width, label=metric_name)

    axis.axhline(0, color='black', linewidth=0.8, alpha=0.5)
    axis.set_xticks(x)
    axis.set_xticklabels(summary['variant_label'], rotation=12, ha='right')
    y_min = min(-0.05, float(summary[['C_txt', 'Leak', 'ContentMargin']].min().min()) * 1.15)
    y_max = max(0.35, float(summary[['C_txt', 'Leak', 'ContentMargin']].max().max()) * 1.15)
    axis.set_ylim(y_min, y_max)
    axis.set_ylabel('CLIP cosine similarity / margin')
    axis.set_title('Notebook 18 content preservation and style-object leakage')
    axis.grid(axis='y', alpha=0.25)
    axis.legend()

    fig.tight_layout()
    plot_path = EVAL_DIR / 'notebook18_content_leakage_metrics_summary.png'
    fig.savefig(plot_path, dpi=180, bbox_inches='tight')
    print('Saved content/leakage metric plot:', plot_path)
    plt.show()

    return metrics_df


RUN_CONTENT_LEAKAGE_METRICS = True

if RUN_CONTENT_LEAKAGE_METRICS:
    CONTENT_LEAKAGE_METRICS_DF = compute_content_leakage_metrics(PAIR_ROWS)
else:
    print('Set RUN_CONTENT_LEAKAGE_METRICS = True after generation to compute C_txt, Leak, and ContentMargin.')


## Download Outputs


In [ ]:
import zipfile


def package_notebook18_outputs_for_download(include_traces=False):
    zip_path = OUTPUT_DIR / 'notebook18_diagnostic_selected_outputs_and_metrics.zip'
    files_to_add = []
    image_dirs = [
        BASELINE_DIR,
        MASK_DIR,
        MASK_OVERLAY_DIR,
        STYLE_MASK_DIR,
        STYLE_MASK_OVERLAY_DIR,
        SCALE_SELECTION_DIR,
        IMPROVED_DIR,
        DIAGNOSTIC_DIR,
        GALLERY_DIR,
        EVAL_DIR,
    ]
    for directory in image_dirs:
        files_to_add.extend(sorted(directory.glob('*.png')))
        files_to_add.extend(sorted(directory.glob('*.jpg')))
        files_to_add.extend(sorted(directory.glob('*.jpeg')))
        files_to_add.extend(sorted(directory.glob('*.csv')))

    files_to_add.extend(sorted(TRACE_DIR.glob('*.json')))
    if include_traces:
        files_to_add.extend(sorted(TRACE_DIR.glob('*.pt')))

    files_to_add = sorted(set(files_to_add))
    if not files_to_add:
        raise FileNotFoundError('No notebook 18 outputs found yet. Run the generation, gallery, diagnostics, and metric cells first.')

    with zipfile.ZipFile(zip_path, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
        for file_path in files_to_add:
            archive.write(file_path, arcname=file_path.relative_to(OUTPUT_DIR))

    print(f'Packaged {len(files_to_add)} files:', zip_path)
    print('Included directories:')
    for directory in image_dirs:
        print(' -', directory.relative_to(OUTPUT_DIR))

    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception:
        print('Download helper is automatic only in Colab. Local zip path:')
        print(zip_path)
    return zip_path

package_notebook18_outputs_for_download(include_traces=False)
